In [ ]:
import requests
import pandas as pd

In [32]:
API_URL = "https://api.hyperliquid.xyz/info"

# Known HIP-3 DEX namespaces (community DEXes)
HIP3_DEXES = ["xyz", "flx", "vntl", "hyna", "km", "cash", "purp", "mfer"]

def get_native_markets():
    """Pull all native/core markets from Hyperliquid."""
    payload = {"type": "metaAndAssetCtxs"}
    r = requests.post(API_URL, json=payload)
    r.raise_for_status()
    meta, asset_ctxs = r.json()

    markets = []
    for i, asset in enumerate(meta["universe"]):
        # Skip delisted
        if asset.get("isDelisted", False):
            continue
            
        ctx = asset_ctxs[i]
        markets.append({
            "ticker": asset["name"],
            "dex": "native",
            "market_type": "native",
            "volume_24h_usd": float(ctx.get("dayNtlVlm", 0)),
            "open_interest_usd": float(ctx.get("openInterest", 0)),
            "price": float(ctx.get("markPx", 0)),
            "funding_rate": float(ctx.get("funding", 0)),
            "max_leverage": asset.get("maxLeverage", 0)
        })
    return markets

def get_dex_markets(dex_name):
    """Pull all markets from a HIP-3 DEX."""
    payload = {"type": "metaAndAssetCtxs", "dex": dex_name}
    r = requests.post(API_URL, json=payload)
    r.raise_for_status()
    meta, asset_ctxs = r.json()

    markets = []
    for i, asset in enumerate(meta["universe"]):
        # Skip delisted
        if asset.get("isDelisted", False):
            continue
            
        ctx = asset_ctxs[i]
        ticker = asset['name']
        
        markets.append({
            "ticker": ticker,
            "dex": dex_name,
            "market_type": "hip3",
            "volume_24h_usd": float(ctx.get("dayNtlVlm", 0)),
            "open_interest_usd": float(ctx.get("openInterest", 0)),
            "price": float(ctx.get("markPx", 0)),
            "funding_rate": float(ctx.get("funding", 0)),
            "max_leverage": asset.get("maxLeverage", 0)
        })
    return markets

def get_all_markets_with_stats(top_n=1000):
    """Aggregate native + all HIP-3 markets with stats."""
    all_markets = []

    print("Fetching native markets...")
    try:
        native_markets = get_native_markets()
        all_markets.extend(native_markets)
        print(f"Found {len(native_markets)} native markets")
    except Exception as e:
        print(f"Failed to fetch native markets: {e}")

    print(f"\nFetching HIP-3 markets from {len(HIP3_DEXES)} DEXes...")
    print(f"  DEXes to try: {', '.join(HIP3_DEXES)}")
    
    total_hip3_markets = 0
    active_dexes = []
    
    for dex in HIP3_DEXES:
        try:
            print(f"\n  Fetching {dex} DEX...")
            dex_markets = get_dex_markets(dex)
            if len(dex_markets) > 0:
                all_markets.extend(dex_markets)
                total_hip3_markets += len(dex_markets)
                active_dexes.append(dex)
                print(f"Found {len(dex_markets)} markets in {dex}")
            else:
                print(f"No markets in {dex}")
        except Exception as e:
            print(f"Failed to fetch {dex}: {e}")

    print(f"\n  Summary: {total_hip3_markets} HIP-3 markets from {len(active_dexes)} active DEXes")
    print(f"Active DEXes: {', '.join(active_dexes)}")

    if not all_markets:
        print("\nNo markets found!")
        return pd.DataFrame()
    
    df = pd.DataFrame(all_markets)
    df = df.sort_values("volume_24h_usd", ascending=False)
    
    print(f"\nTotal markets fetched: {len(df)}")
    return df.head(top_n)


In [ ]:
# Get all markets
top_markets = get_all_markets_with_stats(top_n=1000)


In [34]:
top_50 = 50

print("\n" + "=" * 80)
print("TOP HYPERLIQUID MARKETS BY 24H VOLUME")
print("=" * 80)
print(f"\n{'Rank':<6}{'Ticker':<25}{'DEX':<10}{'Type':<10}{'24h Volume':<18}{'Open Interest':<18}{'Price':<12}")
print("-" * 100)

for idx, (_, row) in enumerate(top_markets.head(50).iterrows(), 1):
    ticker = row['ticker']
    dex = row['dex']
    market_type = row['market_type']
    volume = f"${row['volume_24h_usd']:,.0f}"
    oi = f"${row['open_interest_usd']:,.0f}"
    price = f"${row['price']:,.2f}"
    
    print(f"{idx:<6}{ticker:<25}{dex:<10}{market_type:<10}{volume:<18}{oi:<18}{price:<12}")

# Summary statistics
print("\n" + "=" * 100)
print("SUMMARY STATISTICS")
print("=" * 100)

native_df = top_markets[top_markets['market_type'] == 'native']
hip3_df = top_markets[top_markets['market_type'] == 'hip3']

print(f"\nTotal markets (active): {len(top_markets)}")
print(f"Native markets: {len(native_df)}")
print(f"HIP-3 markets: {len(hip3_df)}")

print(f"\nTotal 24h volume: ${top_markets['volume_24h_usd'].sum():,.0f}")
print(f"  Native markets: ${native_df['volume_24h_usd'].sum():,.0f}")
print(f"  HIP-3 markets: ${hip3_df['volume_24h_usd'].sum():,.0f}")

print(f"\nTotal open interest: ${top_markets['open_interest_usd'].sum():,.0f}")
print(f"  Native markets: ${native_df['open_interest_usd'].sum():,.0f}")
print(f"  HIP-3 markets: ${hip3_df['open_interest_usd'].sum():,.0f}")

# Breakdown by DEX
print("\n" + "=" * 100)
print("BREAKDOWN BY DEX")
print("=" * 100)
dex_stats = top_markets.groupby('dex').agg({
    'ticker': 'count',
    'volume_24h_usd': 'sum',
    'open_interest_usd': 'sum'
}).rename(columns={'ticker': 'markets'})
dex_stats = dex_stats.sort_values('volume_24h_usd', ascending=False)

print(f"\n{'DEX':<10}{'Markets':<10}{'24h Volume':<20}{'Open Interest':<20}")
print("-" * 60)
for dex, row in dex_stats.iterrows():
    print(f"{dex:<10}{int(row['markets']):<10}${row['volume_24h_usd']:>15,.0f}   ${row['open_interest_usd']:>15,.0f}")

# Show top 10 native markets
print("\n" + "=" * 100)
print(f"TOP {top_50} NATIVE MARKETS")
print("=" * 100)
for idx, (_, row) in enumerate(native_df.head(top_50).iterrows(), 1):
    print(f"{idx}. {row['ticker']:<10} Vol: ${row['volume_24h_usd']:>15,.0f}  OI: ${row['open_interest_usd']:>15,.0f}  Price: ${row['price']:>10,.2f}")

# Show top 10 HIP-3 markets
print("\n" + "=" * 100)
print(f"TOP {top_50} HIP-3 (COMMUNITY) MARKETS")
print("=" * 100)
if len(hip3_df) > 0:
    for idx, (_, row) in enumerate(hip3_df.head(top_50).iterrows(), 1):
        print(f"{idx}. {row['ticker']:<25} ({row['dex']:<4}) Vol: ${row['volume_24h_usd']:>12,.0f}  OI: ${row['open_interest_usd']:>12,.0f}  Price: ${row['price']:>10,.6f}")
else:
    print("No HIP-3 markets found")

# Show top markets per HIP-3 DEX
print("\n" + "=" * 100)
print("TOP MARKET FROM EACH HIP-3 DEX")
print("=" * 100)
for dex in hip3_df['dex'].unique():
    dex_markets = hip3_df[hip3_df['dex'] == dex].head(1)
    if len(dex_markets) > 0:
        row = dex_markets.iloc[0]
        print(f"{dex:<6} → {row['ticker']:<25} Vol: ${row['volume_24h_usd']:>12,.0f}  OI: ${row['open_interest_usd']:>12,.0f}")


TOP HYPERLIQUID MARKETS BY 24H VOLUME

Rank  Ticker                   DEX       Type      24h Volume        Open Interest     Price       
----------------------------------------------------------------------------------------------------
1     BTC                      native    native    $3,545,184,112    $21,152           $67,077.00  
2     ETH                      native    native    $1,170,875,942    $616,978          $2,004.10   
3     SOL                      native    native    $442,894,351      $3,539,404        $86.50      
4     HYPE                     native    native    $328,348,916      $18,251,468       $31.00      
5     xyz:SILVER               xyz       hip3      $189,014,560      $1,813,686        $94.36      
6     xyz:GOLD                 xyz       hip3      $163,346,207      $38,946           $5,330.70   
7     XRP                      native    native    $89,444,126       $54,527,286       $1.40       
8     xyz:XYZ100               xyz       hip3      $84,919,